In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CATEGORIES
from src.utils.io import read_jsonl_sample

CATEGORY = 'romance'
cfg = CATEGORIES[CATEGORY]
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Category: {cfg.display_name}')
print(f'Books file: {cfg.books_file}')
print(f'Interactions file: {cfg.interactions_file}')
print(f'Output dir: {OUTPUT_DIR}')


Category: Romance
Books file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_books_romance.json.gz
Interactions file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_interactions_romance.json.gz
Output dir: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\interim\romance


---
# Books

In [2]:
SAMPLE_SIZE = 5_000
books_sample = read_jsonl_sample(cfg.books_file, nrows=SAMPLE_SIZE)
print(f'Shape: {books_sample.shape}')
print(f'Columns ({len(books_sample.columns)}):')
print(list(books_sample.columns))

Shape: (5000, 29)
Columns (29):
['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']


In [3]:
summary_books = pd.DataFrame({
    'dtype': books_sample.dtypes.astype(str),
    'non_null': books_sample.notna().sum(),
    'null_count': books_sample.isna().sum(),
    'null_pct': (books_sample.isna().sum() / len(books_sample) * 100).round(1),
    'n_unique': books_sample.apply(lambda col: col.map(str).nunique()),
    'example': books_sample.iloc[0].astype(str).str[:80],
})
display(summary_books)

,dtype,non_null,null_count,null_pct,n_unique,example
isbn,str,5000,0,0.0,1914,
text_reviews_count,str,5000,0,0.0,327,4
series,object,5000,0,0.0,3003,[]
country_code,str,5000,0,0.0,1,US
language_code,str,5000,0,0.0,37,
popular_shelves,object,5000,0,0.0,4871,"[{'count': '4', 'name': 'to-read'}, {'count': ..."
asin,str,5000,0,0.0,1837,
is_ebook,str,5000,0,0.0,2,true
average_rating,str,5000,0,0.0,234,3.86
kindle_asin,str,5000,0,0.0,2652,


In [4]:
books_sample.head(3)

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,4,[],US,,"[{'count': '4', 'name': 'to-read'}, {'count': ...",,true,3.86,,...,5,,2017,https://www.goodreads.com/book/show/34883016-p...,https://images.gr-assets.com/books/1493525974m...,34883016,5,56135087,Playmaker: A Venom Series Novella,Playmaker: A Venom Series Novella
1,,21,[811663],US,en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",B01BLJGA9S,true,4.23,B01BLJGA9S,...,,,,https://www.goodreads.com/book/show/29074693-p...,https://s.gr-assets.com/assets/nophoto/book/11...,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)"
2,1597371289,8,[],US,eng,"[{'count': '16215', 'name': 'classics'}, {'cou...",,false,3.99,B0083Z3O8Y,...,9,,2005,https://www.goodreads.com/book/show/3209316-emma,https://s.gr-assets.com/assets/nophoto/book/11...,3209316,42,3360164,Emma,Emma


In [5]:
BOOKS_COLUMNS_TO_DROP = [
    'isbn',
    'isbn13',
    'asin',
    'kindle_asin',
    'url',
    'link',
    'image_url',
    'country_code',
    'edition_information',
    'publication_day',
    'publication_month',
    'similar_books',
    'title_without_series',
    'is_ebook',
    'description',
    'publisher',
    'format'
]

In [6]:
books_clean = books_sample.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in books_sample.columns])

In [7]:
summary_clean = pd.DataFrame({
    'dtype': books_clean.dtypes.astype(str),
    'non_null': books_clean.notna().sum(),
    'null_pct': (books_clean.isna().sum() / len(books_clean) * 100).round(1),
})
display(summary_clean)

,dtype,non_null,null_pct
text_reviews_count,str,5000,0.0
series,object,5000,0.0
language_code,str,5000,0.0
popular_shelves,object,5000,0.0
average_rating,str,5000,0.0
authors,object,5000,0.0
num_pages,str,5000,0.0
publication_year,str,5000,0.0
book_id,str,5000,0.0
ratings_count,str,5000,0.0


In [8]:
books_clean.head(5)

,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,4,[],,"[{'count': '4', 'name': 'to-read'}, {'count': ...",3.86,"[{'author_id': '5807700', 'role': ''}]",,2017,34883016,5,56135087,Playmaker: A Venom Series Novella
1,21,[811663],en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",4.23,"[{'author_id': '5360266', 'role': ''}]",,,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)"
2,8,[],eng,"[{'count': '16215', 'name': 'classics'}, {'cou...",3.99,"[{'author_id': '1265', 'role': ''}]",544,2005,3209316,42,3360164,Emma
3,27,[938303],en-GB,"[{'count': '25', 'name': 'to-read'}, {'count':...",4.31,"[{'author_id': '90411', 'role': ''}, {'author_...",,,30838933,139,51437308,"Guardian Cougar (Finding Fatherhood, #2)"
4,15,[],eng,"[{'count': '1492', 'name': 'to-read'}, {'count...",3.98,"[{'author_id': '47231', 'role': ''}]",,,27419760,167,46003673,Wedding Girl


In [9]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

BOOKS_OUTPUT_PATH = OUTPUT_DIR / 'books_clean.parquet'
CHUNKSIZE = 50_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.books_file, chunksize=CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(BOOKS_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    print(f'  Chunk {chunk_idx}: {len(chunk):,} rows (total: {total_rows:,})')

if writer:
    writer.close()

  Chunk 0: 50,000 rows (total: 50,000)
  Chunk 1: 50,000 rows (total: 100,000)
  Chunk 2: 50,000 rows (total: 150,000)
  Chunk 3: 50,000 rows (total: 200,000)
  Chunk 4: 50,000 rows (total: 250,000)
  Chunk 5: 50,000 rows (total: 300,000)
  Chunk 6: 35,449 rows (total: 335,449)


In [10]:
metadata = pq.read_metadata(BOOKS_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {BOOKS_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(BOOKS_OUTPUT_PATH, columns=None).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 335,449
Columns: 14
File size: 234.2 MB

Final columns (12):
['text_reviews_count', 'series', 'language_code', 'popular_shelves', 'average_rating', 'authors', 'num_pages', 'publication_year', 'book_id', 'ratings_count', 'work_id', 'title']


,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,4,[],,"[{'count': '4', 'name': 'to-read'}, {'count': ...",3.86,"[{'author_id': '5807700', 'role': ''}]",,2017,34883016,5,56135087,Playmaker: A Venom Series Novella
1,21,[811663],en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",4.23,"[{'author_id': '5360266', 'role': ''}]",,,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)"
2,8,[],eng,"[{'count': '16215', 'name': 'classics'}, {'cou...",3.99,"[{'author_id': '1265', 'role': ''}]",544,2005,3209316,42,3360164,Emma


# Interactions

In [11]:
INTER_SAMPLE_SIZE = 10_000
inter_sample = read_jsonl_sample(cfg.interactions_file, nrows=INTER_SAMPLE_SIZE)
print(f'Shape: {inter_sample.shape}')
print(f'Columns ({len(inter_sample.columns)}):')
print(list(inter_sample.columns))

Shape: (10000, 10)
Columns (10):
['user_id', 'book_id', 'review_id', 'is_read', 'rating', 'review_text_incomplete', 'date_added', 'date_updated', 'read_at', 'started_at']


In [12]:
summary_inter = pd.DataFrame({
    'dtype': inter_sample.dtypes.astype(str),
    'non_null': inter_sample.notna().sum(),
    'null_count': inter_sample.isna().sum(),
    'null_pct': (inter_sample.isna().sum() / len(inter_sample) * 100).round(1),
    'n_unique': inter_sample.apply(lambda col: col.map(str).nunique()),
    'example': inter_sample.iloc[0].astype(str).str[:80],
})
display(summary_inter)

,dtype,non_null,null_count,null_pct,n_unique,example
user_id,str,10000,0,0.0,75,8842281e1d1347389f2ab93d60773d4d
book_id,str,10000,0,0.0,7577,5526
review_id,str,10000,0,0.0,10000,c2b5f16df9f9c716617c07326b4f2869
is_read,bool,10000,0,0.0,2,False
rating,int64,10000,0,0.0,6,0
review_text_incomplete,str,10000,0,0.0,1320,
date_added,str,10000,0,0.0,9829,Wed Mar 29 00:27:40 -0700 2017
date_updated,str,10000,0,0.0,9825,Wed Mar 29 00:27:40 -0700 2017
read_at,str,10000,0,0.0,1886,
started_at,str,10000,0,0.0,1485,


In [13]:
inter_sample.head(3)

,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,5526,c2b5f16df9f9c716617c07326b4f2869,False,0,,Wed Mar 29 00:27:40 -0700 2017,Wed Mar 29 00:27:40 -0700 2017,,
1,8842281e1d1347389f2ab93d60773d4d,18633693,611eea84ef983e27884953b8e45b55aa,False,0,,Tue Mar 28 21:01:27 -0700 2017,Tue Mar 28 21:01:27 -0700 2017,,
2,8842281e1d1347389f2ab93d60773d4d,22055137,9f149f1a24f51b01aed5b7e465134027,False,0,,Tue Mar 28 00:02:48 -0700 2017,Tue Mar 28 00:02:49 -0700 2017,,


In [14]:
INTER_COLUMNS_TO_DROP = [
    'review_text_incomplete',
    'review_id',
    'read_at',
    'started_at',
]

In [15]:
inter_clean = inter_sample.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in inter_sample.columns])

In [16]:
summary_inter_clean = pd.DataFrame({
    'dtype': inter_clean.dtypes.astype(str),
    'non_null': inter_clean.notna().sum(),
    'null_pct': (inter_clean.isna().sum() / len(inter_clean) * 100).round(1),
})
display(summary_inter_clean)

,dtype,non_null,null_pct
user_id,str,10000,0.0
book_id,str,10000,0.0
is_read,bool,10000,0.0
rating,int64,10000,0.0
date_added,str,10000,0.0
date_updated,str,10000,0.0


In [17]:
inter_clean.head(5)

,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,5526,False,0,Wed Mar 29 00:27:40 -0700 2017,Wed Mar 29 00:27:40 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,18633693,False,0,Tue Mar 28 21:01:27 -0700 2017,Tue Mar 28 21:01:27 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,22055137,False,0,Tue Mar 28 00:02:48 -0700 2017,Tue Mar 28 00:02:49 -0700 2017
3,8842281e1d1347389f2ab93d60773d4d,23677430,False,0,Mon Mar 27 22:40:46 -0700 2017,Mon Mar 27 22:40:47 -0700 2017
4,8842281e1d1347389f2ab93d60773d4d,23002924,False,0,Mon Mar 27 22:01:41 -0700 2017,Mon Mar 27 22:01:41 -0700 2017


In [18]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

INTER_OUTPUT_PATH = OUTPUT_DIR / 'interactions_clean.parquet'
INTER_CHUNKSIZE = 500_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.interactions_file, chunksize=INTER_CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(INTER_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    
    if chunk_idx % 10 == 0:
        print(f'  Chunk {chunk_idx}: {total_rows:,} rows so far')

if writer:
    writer.close()

print(f'   Total rows: {total_rows:,}')

  Chunk 0: 500,000 rows so far
  Chunk 10: 5,500,000 rows so far
  Chunk 20: 10,500,000 rows so far
  Chunk 30: 15,500,000 rows so far
  Chunk 40: 20,500,000 rows so far
  Chunk 50: 25,500,000 rows so far
  Chunk 60: 30,500,000 rows so far
  Chunk 70: 35,500,000 rows so far
  Chunk 80: 40,500,000 rows so far
   Total rows: 42,792,856


In [19]:
metadata = pq.read_metadata(INTER_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {INTER_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(INTER_OUTPUT_PATH).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 42,792,856
Columns: 6
File size: 1051.8 MB

Final columns (6):
['user_id', 'book_id', 'is_read', 'rating', 'date_added', 'date_updated']


,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,5526,False,0,Wed Mar 29 00:27:40 -0700 2017,Wed Mar 29 00:27:40 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,18633693,False,0,Tue Mar 28 21:01:27 -0700 2017,Tue Mar 28 21:01:27 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,22055137,False,0,Tue Mar 28 00:02:48 -0700 2017,Tue Mar 28 00:02:49 -0700 2017
